In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"7marchecht","vettedcellcounts5.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"8marchecht", "UMAP_5")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=1100, 
        height=800,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#alles, engkel aggreagatd zonder preprocessing

In [ ]:

import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "7marchecht", "aggregated_wells_data_cellcount0.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_alleszonderfeatureselect")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

df.columns = [str(c) for c in df.columns]

# Data Parsing
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# --- 2. COLOR SCHEMES ---
unique_combos = sorted(df['Plate'].unique())

# PLATE VIEW: Turbo palette for high-contrast distinction between all batches
turbo_colors = px.colors.sample_colorscale("Turbo", [i/(len(unique_combos)-1) for i in range(len(unique_combos))])
combo_color_map = {combo: turbo_colors[i] for i, combo in enumerate(unique_combos)}

# TIME VIEW: High-contrast Blue Gradient (Light Blue -> Mid Blue -> Deep Navy)
time_colors = {
    'T0': '#A9D1FF', # Visible Light Blue (Sky)
    'T1': '#2A7FFF', # Strong Primary Blue
    'T2': '#001A4D'  # Deep Midnight Navy
}

def darken_color(hex_color, factor=0.4):
    """Significantly deepens the color for square controls."""
    if hex_color.startswith('rgb'):
        rgb = [int(x) for x in hex_color[4:-1].split(',')]
    else:
        hex_color = hex_color.lstrip('#')
        rgb = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    
    dark_rgb = tuple(max(0, int(c * factor)) for c in rgb)
    return f'rgb{dark_rgb}'

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION ---
for config in plot_configs:
    if not config['indices']: continue
    print(f"Generating Plots for: {config['name']}...")
    
    # UMAP Computation
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # --- PLOT 1: PLATE-TIME VIEW ---
    fig_plate = go.Figure()
    for combo in unique_combos:
        for t_type in ['Mutant', 'Control']:
            mask = (df_plot['Plate'] == combo) & (df_plot['Type'] == t_type)
            curr = df_plot[mask]
            if curr.empty: continue
            
            base_col = combo_color_map[combo]
            final_col = base_col if t_type == 'Mutant' else darken_color(base_col)
            
            fig_plate.add_trace(go.Scatter(
                x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                name=f"{combo}",
                marker=dict(
                    color=final_col, 
                    size=8 if t_type == 'Mutant' else 11,
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0),
                    opacity=0.8
                ),
                customdata=np.stack((curr['Plate'], curr['Well_ID'], curr['Treatment'], curr['Cell_Count']), axis=-1),
                hovertemplate=(
                    "<b>%{customdata[2]}</b><br>" +
                    "Plate: %{customdata[0]}<br>" +
                    "Well: %{customdata[1]}<br>" +
                    "Count: %{customdata[3]}<extra></extra>"
                ),
                showlegend=True if t_type == 'Mutant' else False,
                legendgroup=combo
            ))

    fig_plate.update_layout(
        title=f"Plate-Time View: {config['name']}",
        template='plotly_white',
        width=850, height=850,
        xaxis=dict(title="UMAP 1", showticklabels=False),
        yaxis=dict(title="UMAP 2", showticklabels=False)
    )
    fig_plate.write_html(os.path.join(OUTPUT_DIR, f"{config['name']}_PLATE_VIEW.html"))

    # --- PLOT 2: TIMEPOINT VIEW (BLUE GRADIENT) ---
    fig_time = go.Figure()
    for time in sorted(df_plot['Timepoint'].unique()):
        for t_type in ['Mutant', 'Control']:
            mask = (df_plot['Timepoint'] == time) & (df_plot['Type'] == t_type)
            curr = df_plot[mask]
            if curr.empty: continue
            
            fig_time.add_trace(go.Scatter(
                x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                name=time,
                marker=dict(
                    color=time_colors[time], 
                    size=7 if t_type == 'Mutant' else 10,
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    line=dict(width=0.8, color='black') if t_type == 'Control' else dict(width=0),
                    opacity=0.75
                ),
                customdata=np.stack((curr['Plate'], curr['Well_ID'], curr['Treatment'], curr['Cell_Count']), axis=-1),
                hovertemplate=(
                    "<b>%{customdata[2]}</b><br>" +
                    "Plate: %{customdata[0]}<br>" +
                    "Well: %{customdata[1]}<br>" +
                    "Count: %{customdata[3]}<extra></extra>"
                ),
                showlegend=True if t_type == 'Mutant' else False,
                legendgroup=time
            ))

    fig_time.update_layout(
        title=f"Timepoint View: {config['name']}",
        template='plotly_white',
        width=850, height=850,
        xaxis=dict(title="UMAP 1", showticklabels=False),
        yaxis=dict(title="UMAP 2", showticklabels=False)
    )
    fig_time.write_html(os.path.join(OUTPUT_DIR, f"{config['name']}_TIME_VIEW.html"))

    print(f"Completed and saved: {config['name']}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "7marchecht", "aggregated_wells_data_cellcount0.csv")
df = pd.read_csv(file_path)

# Separate directories for HTML and SVG
HTML_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_allesInteractive")
SVG_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_allesVector_Scalable")

for folder in [HTML_DIR, SVG_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

df.columns = [str(c) for c in df.columns]

# Data Parsing
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# --- 2. COLOR SCHEMES ---
unique_combos = sorted(df['Plate'].unique())
turbo_colors = px.colors.sample_colorscale("Turbo", [i/(len(unique_combos)-1) for i in range(len(unique_combos))])
combo_color_map = {combo: turbo_colors[i] for i, combo in enumerate(unique_combos)}

time_colors = {
    'T0': '#A9D1FF', 
    'T1': '#2A7FFF', 
    'T2': '#001A4D'  
}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION ---
for config in plot_configs:
    if not config['indices']: continue
    print(f"Generating Plots and SVGs for: {config['name']}...")
    
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # --- PLOT LOOP (Plate & Time) ---
    modes = [
        ('PLATE', unique_combos, combo_color_map, 'Plate-Time View'),
        ('TIME', sorted(df_plot['Timepoint'].unique()), time_colors, 'Timepoint View')
    ]

    for mode_name, groups, color_map, title_prefix in modes:
        fig = go.Figure()
        
        for group in groups:
            for t_type in ['Mutant', 'Control']:
                # Determine filtering logic based on mode
                mask = (df_plot['Plate' if mode_name == 'PLATE' else 'Timepoint'] == group) & (df_plot['Type'] == t_type)
                curr = df_plot[mask]
                if curr.empty: continue
                
                color = color_map[group]
                
                fig.add_trace(go.Scatter(
                    x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                    name=str(group),
                    marker=dict(
                        color=color, 
                        size=8 if t_type == 'Mutant' else 11,
                        symbol='circle' if t_type == 'Mutant' else 'square',
                        line=dict(width=1.0, color='black') if t_type == 'Control' else dict(width=0),
                        opacity=1.0
                    ),
                    customdata=np.stack((curr['Plate'], curr['Well_ID'], curr['Treatment'], curr['Cell_Count']), axis=-1),
                    hovertemplate="<b>%{customdata[2]}</b><br>Plate: %{customdata[0]}<br>Well: %{customdata[1]}<br>Count: %{customdata[3]}<extra></extra>",
                    showlegend=True if t_type == 'Mutant' else False,
                    legendgroup=str(group)
                ))

        fig.update_layout(
            title=f"{title_prefix}: {config['name']}",
            template='plotly_white',
            width=850, height=850,
            xaxis=dict(title="UMAP 1", showgrid=False, showticklabels=False),
            yaxis=dict(title="UMAP 2", showgrid=False, showticklabels=False)
        )

        # 1. Save HTML
        html_name = f"{config['name'].replace(' ', '_')}_{mode_name}.html"
        fig.write_html(os.path.join(HTML_DIR, html_name))
        
        # 2. Save SVG (requires kaleido)
        svg_name = f"{config['name'].replace(' ', '_')}_{mode_name}.svg"
        fig.write_image(os.path.join(SVG_DIR, svg_name))

    print(f"Successfully saved {config['name']} exports.")

print("\nAll files (HTML and SVG) are ready in the output folders.")

In [ ]:
#time and plate sperate, preprocessing and less than 5 cells removed

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_Standalone_Blue")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

df.columns = [str(c) for c in df.columns]

# Data Parsing
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# --- 2. COLOR SCHEMES ---
unique_combos = sorted(df['Plate'].unique())

# PLATE VIEW: Turbo palette for high-contrast distinction between all batches
turbo_colors = px.colors.sample_colorscale("Turbo", [i/(len(unique_combos)-1) for i in range(len(unique_combos))])
combo_color_map = {combo: turbo_colors[i] for i, combo in enumerate(unique_combos)}

# TIME VIEW: High-contrast Blue Gradient (Light Blue -> Mid Blue -> Deep Navy)
time_colors = {
    'T0': '#A9D1FF', # Visible Light Blue (Sky)
    'T1': '#2A7FFF', # Strong Primary Blue
    'T2': '#001A4D'  # Deep Midnight Navy
}

def darken_color(hex_color, factor=0.4):
    """Significantly deepens the color for square controls."""
    if hex_color.startswith('rgb'):
        rgb = [int(x) for x in hex_color[4:-1].split(',')]
    else:
        hex_color = hex_color.lstrip('#')
        rgb = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    
    dark_rgb = tuple(max(0, int(c * factor)) for c in rgb)
    return f'rgb{dark_rgb}'

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION ---
for config in plot_configs:
    if not config['indices']: continue
    print(f"Generating Plots for: {config['name']}...")
    
    # UMAP Computation
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # --- PLOT 1: PLATE-TIME VIEW ---
    fig_plate = go.Figure()
    for combo in unique_combos:
        for t_type in ['Mutant', 'Control']:
            mask = (df_plot['Plate'] == combo) & (df_plot['Type'] == t_type)
            curr = df_plot[mask]
            if curr.empty: continue
            
            base_col = combo_color_map[combo]
            final_col = base_col if t_type == 'Mutant' else darken_color(base_col)
            
            fig_plate.add_trace(go.Scatter(
                x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                name=f"{combo}",
                marker=dict(
                    color=final_col, 
                    size=8 if t_type == 'Mutant' else 11,
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0),
                    opacity=0.8
                ),
                customdata=np.stack((curr['Plate'], curr['Well_ID'], curr['Treatment'], curr['Cell_Count']), axis=-1),
                hovertemplate=(
                    "<b>%{customdata[2]}</b><br>" +
                    "Plate: %{customdata[0]}<br>" +
                    "Well: %{customdata[1]}<br>" +
                    "Count: %{customdata[3]}<extra></extra>"
                ),
                showlegend=True if t_type == 'Mutant' else False,
                legendgroup=combo
            ))

    fig_plate.update_layout(
        title=f"Plate-Time View: {config['name']}",
        template='plotly_white',
        width=850, height=850,
        xaxis=dict(title="UMAP 1", showticklabels=False),
        yaxis=dict(title="UMAP 2", showticklabels=False)
    )
    fig_plate.write_html(os.path.join(OUTPUT_DIR, f"{config['name']}_PLATE_VIEW.html"))

    # --- PLOT 2: TIMEPOINT VIEW (BLUE GRADIENT) ---
    fig_time = go.Figure()
    for time in sorted(df_plot['Timepoint'].unique()):
        for t_type in ['Mutant', 'Control']:
            mask = (df_plot['Timepoint'] == time) & (df_plot['Type'] == t_type)
            curr = df_plot[mask]
            if curr.empty: continue
            
            fig_time.add_trace(go.Scatter(
                x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                name=time,
                marker=dict(
                    color=time_colors[time], 
                    size=7 if t_type == 'Mutant' else 10,
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    line=dict(width=0.8, color='black') if t_type == 'Control' else dict(width=0),
                    opacity=0.75
                ),
                customdata=np.stack((curr['Plate'], curr['Well_ID'], curr['Treatment'], curr['Cell_Count']), axis=-1),
                hovertemplate=(
                    "<b>%{customdata[2]}</b><br>" +
                    "Plate: %{customdata[0]}<br>" +
                    "Well: %{customdata[1]}<br>" +
                    "Count: %{customdata[3]}<extra></extra>"
                ),
                showlegend=True if t_type == 'Mutant' else False,
                legendgroup=time
            ))

    fig_time.update_layout(
        title=f"Timepoint View: {config['name']}",
        template='plotly_white',
        width=850, height=850,
        xaxis=dict(title="UMAP 1", showticklabels=False),
        yaxis=dict(title="UMAP 2", showticklabels=False)
    )
    fig_time.write_html(os.path.join(OUTPUT_DIR, f"{config['name']}_TIME_VIEW.html"))

    print(f"Completed and saved: {config['name']}")

In [ ]:
pip install --upgrade kaleido

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5.csv")
df = pd.read_csv(file_path)

# Separate directories for HTML and SVG
HTML_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_preprosessInteractive")
SVG_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_preprosessVector_Scalable")

for folder in [HTML_DIR, SVG_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

df.columns = [str(c) for c in df.columns]

# Data Parsing
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# --- 2. COLOR SCHEMES ---
unique_combos = sorted(df['Plate'].unique())
turbo_colors = px.colors.sample_colorscale("Turbo", [i/(len(unique_combos)-1) for i in range(len(unique_combos))])
combo_color_map = {combo: turbo_colors[i] for i, combo in enumerate(unique_combos)}

time_colors = {
    'T0': '#A9D1FF', 
    'T1': '#2A7FFF', 
    'T2': '#001A4D'  
}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION ---
for config in plot_configs:
    if not config['indices']: continue
    print(f"Generating Plots and SVGs for: {config['name']}...")
    
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # --- PLOT LOOP (Plate & Time) ---
    modes = [
        ('PLATE', unique_combos, combo_color_map, 'Plate-Time View'),
        ('TIME', sorted(df_plot['Timepoint'].unique()), time_colors, 'Timepoint View')
    ]

    for mode_name, groups, color_map, title_prefix in modes:
        fig = go.Figure()
        
        for group in groups:
            for t_type in ['Mutant', 'Control']:
                # Determine filtering logic based on mode
                mask = (df_plot['Plate' if mode_name == 'PLATE' else 'Timepoint'] == group) & (df_plot['Type'] == t_type)
                curr = df_plot[mask]
                if curr.empty: continue
                
                color = color_map[group]
                
                fig.add_trace(go.Scatter(
                    x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                    name=str(group),
                    marker=dict(
                        color=color, 
                        size=8 if t_type == 'Mutant' else 11,
                        symbol='circle' if t_type == 'Mutant' else 'square',
                        line=dict(width=1.0, color='black') if t_type == 'Control' else dict(width=0),
                        opacity=1.0
                    ),
                    customdata=np.stack((curr['Plate'], curr['Well_ID'], curr['Treatment'], curr['Cell_Count']), axis=-1),
                    hovertemplate="<b>%{customdata[2]}</b><br>Plate: %{customdata[0]}<br>Well: %{customdata[1]}<br>Count: %{customdata[3]}<extra></extra>",
                    showlegend=True if t_type == 'Mutant' else False,
                    legendgroup=str(group)
                ))

        fig.update_layout(
            title=f"{title_prefix}: {config['name']}",
            template='plotly_white',
            width=850, height=850,
            xaxis=dict(title="UMAP 1", showgrid=False, showticklabels=False),
            yaxis=dict(title="UMAP 2", showgrid=False, showticklabels=False)
        )

        # 1. Save HTML
        html_name = f"{config['name'].replace(' ', '_')}_{mode_name}.html"
        fig.write_html(os.path.join(HTML_DIR, html_name))
        
        # 2. Save SVG (requires kaleido)
        svg_name = f"{config['name'].replace(' ', '_')}_{mode_name}.svg"
        fig.write_image(os.path.join(SVG_DIR, svg_name))

    print(f"Successfully saved {config['name']} exports.")

print("\nAll files (HTML and SVG) are ready in the output folders.")

In [ ]:
#annotation

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# --- 1. SETUP ---
# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script


OUTPUT_DIR = os.path.join(PROJECT_ROOT,"8marchecht", "UMAP_5_annotated")

# Create the output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT,"7marchecht","vettedcellcounts5.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. MERGE & FIX ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['SubtiWiki Annotation 4'] = df['SubtiWiki Annotation 4'].fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

df['Display_Category'] = df['SubtiWiki Annotation 4']
df.loc[df['Timepoint'] == 'T0', 'Display_Category'] = 'Baseline (T0)'

# --- 3. DYNAMIC COLOR MAPPING ---
all_cats = df['Display_Category'].unique()
standard_colors = px.colors.qualitative.Alphabet 

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['Baseline (T0)'] = 'lightgrey'
color_map['Unknown/Other'] = 'black'

# --- 4. RUN UMAP LOOP AND SAVE ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280) + get_channel_features(3840, 5120)},
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Generating and Saving UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Timepoint',
        symbol_map={"T0": "circle", "T1": "x", "T2": "circle"},
        hover_name='Treatment',
        hover_data=['Plate', 'SubtiWiki Annotation 4'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    fig.update_traces(marker=dict(opacity=1.0)) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=6), selector=dict(marker_symbol='x'))
    
    fig.update_layout(width=1100, height=850)
    
    # --- SAVE TO LOCATION ---
    save_path = os.path.join(OUTPUT_DIR, f"UMAP14_{config['name']}.html")
    fig.write_html(save_path)
    
    # Optional: also show in the notebook
    fig.show()

print(f"\nAll interactive plots have been saved to: {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# --- 1. SETUP ---
# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script


OUTPUT_DIR = os.path.join(PROJECT_ROOT,"8marchecht", "UMAP_5_annotatedwiki3")

# Create the output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT,"7marchecht","vettedcellcounts5.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. MERGE & FIX ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3']], on='Treatment', how='left')
df['SubtiWiki Annotation 3'] = df['SubtiWiki Annotation 3'].fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

df['Display_Category'] = df['SubtiWiki Annotation 3']
df.loc[df['Timepoint'] == 'T0', 'Display_Category'] = 'Baseline (T0)'

# --- 3. DYNAMIC COLOR MAPPING ---
all_cats = df['Display_Category'].unique()
standard_colors = px.colors.qualitative.Alphabet 

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['Baseline (T0)'] = 'lightgrey'
color_map['Unknown/Other'] = 'black'

# --- 4. RUN UMAP LOOP AND SAVE ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280) + get_channel_features(3840, 5120)},
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Generating and Saving UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Timepoint',
        symbol_map={"T0": "circle", "T1": "x", "T2": "circle"},
        hover_name='Treatment',
        hover_data=['Plate', 'SubtiWiki Annotation 3'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    fig.update_traces(marker=dict(opacity=1.0)) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=6), selector=dict(marker_symbol='x'))
    
    fig.update_layout(width=1100, height=850)
    
    # --- SAVE TO LOCATION ---
    save_path = os.path.join(OUTPUT_DIR, f"UMAP14_{config['name']}.html")
    fig.write_html(save_path)
    
    # Optional: also show in the notebook
    fig.show()

print(f"\nAll interactive plots have been saved to: {OUTPUT_DIR}")

In [ ]:
#annotation 3 en 4 samen

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_5_annotated")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. MERGE & FALLBACK ANNOTATIONS ---
# Merge both columns
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

# FALLBACK: Use Annotation 4, if empty use Annotation 3, if still empty use "Unknown/Other"
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3'])
df['Effective_Annotation'] = df['Effective_Annotation'].fillna("Unknown/Other")

df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Define the Display Category
df['Display_Category'] = df['Effective_Annotation']
df.loc[df['Timepoint'] == 'T0', 'Display_Category'] = 'Baseline (T0)'

# --- 3. DYNAMIC COLOR MAPPING ---
all_cats = df['Display_Category'].unique()
# Alphabet + Dark24 to ensure enough colors for the combined annotations
standard_colors = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['Baseline (T0)'] = 'lightgrey'
color_map['Unknown/Other'] = 'black'

# --- 4. RUN UMAP LOOP ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280) + get_channel_features(3840, 5120)},
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Generating UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Timepoint',
        symbol_map={"T0": "circle", "T1": "x", "T2": "circle"},
        hover_name='Treatment',
        hover_data=['Plate', 'Effective_Annotation'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    # Clean up the legend: remove the ", T1" or ", T2" suffixes
    # and hide T0/T2 from the legend if T1 is already showing the color
    fig.for_each_trace(lambda t: t.update(name=t.name.split(",")[0]))
    
    # Marker styling
    fig.update_traces(marker=dict(opacity=1.0)) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='x'))
    
    # Layout and Legend Key
    fig.update_layout(
        width=1200, height=850,
        legend_title_text='Pathway',
        annotations=[
            dict(
                text="<b>Symbol Key:</b> Cross (x) = T1 | Dot (●) = T2 & T0",
                showarrow=False,
                xref="paper", yref="paper",
                x=0.5, y=1.05, # Positioned above the plot
                font=dict(size=12)
            )
        ]
    )
    
    # Save
    save_path = os.path.join(OUTPUT_DIR, f"UMAP_Annotated_{config['name']}.html")
    fig.write_html(save_path)
    fig.show()

print(f"\nAll plots saved to: {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_5_annotated3and4")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. MERGE & FALLBACK ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

# FALLBACK: Annotation 4 -> Annotation 3 -> Unknown
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3'])
df['Effective_Annotation'] = df['Effective_Annotation'].fillna("Unknown/Other")

df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Define the Display Category
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']

# Set Baseline/Control logic
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# --- 3. DYNAMIC COLOR MAPPING ---
# Combine multiple palettes to ensure maximum distinction between pathways (avoids too many purples)
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
standard_colors = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['no_sgrna'] = 'lightgrey'
color_map['Baseline (T0)'] = '#D3D3D3' # Static light grey
color_map['Unknown/Other'] = 'black'

# --- 4. EXECUTION FOR ALL CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']} ({len(config['indices'])} features)...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Define shapes
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Treatment',
        hover_data=['Plate', 'Effective_Annotation'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    # Legend De-duplication
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Styling
    fig.update_traces(marker=dict(opacity=1.0)) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='x'))
    fig.update_traces(marker=dict(size=9, line=dict(width=1, color='black')), selector=dict(marker_symbol='square'))
    
    # --- AXIS AND LAYOUT FIX ---
    fig.update_layout(
        width=1300, height=850,
        legend_title_text='Pathway / Group',
        annotations=[
            dict(
                text="<b>Key:</b> Square = no_sgrna | Cross (x) = T1 | Dot (●) = T2/T0",
                showarrow=False, xref="paper", yref="paper",
                x=0.5, y=1.07, font=dict(size=14),
                bgcolor="white", bordercolor="black", borderwidth=1
            )
        ],
        xaxis=dict(
            title="UMAP 1", 
            showline=True, linewidth=2, linecolor='black', mirror=False, 
            showgrid=False, zeroline=False # REMOVED LIGHT GREY AXIS
        ),
        yaxis=dict(
            title="UMAP 2", 
            showline=True, linewidth=2, linecolor='black', mirror=False, 
            showgrid=False, zeroline=False # REMOVED LIGHT GREY AXIS
        )
    )
    
    # Save
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{file_base}.svg"))

print(f"Done. All channels processed with clean axes and high-contrast colors.")

In [ ]:
#1tijdspunt

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_5_annotatedT1")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. PLATE/TIME FILTERING ---
# SPECIFY YOUR PLATES HERE. Leave as None or [] to plot everything.
SELECTED_PLATES = ["PLATE1_T1", "PLATE2_T1", "PLATE3_T1","PLATE4_T1","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()
    print(f"Filtering complete. Plotting {len(df)} cells from {len(SELECTED_PLATES)} plates.")

# --- 3. MERGE & FALLBACK ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

# FALLBACK: Annotation 4 -> Annotation 3 -> Unknown
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3'])
df['Effective_Annotation'] = df['Effective_Annotation'].fillna("Unknown/Other")

df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Define the Display Category
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']

# Set Baseline/Control logic
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# --- 4. DYNAMIC COLOR MAPPING ---
# Expanded palette to avoid purple confusion
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
standard_colors = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['no_sgrna'] = 'lightgrey'
color_map['Baseline (T0)'] = '#D3D3D3'
color_map['Unknown/Other'] = 'black'

# --- 5. EXECUTION FOR ALL CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Define shapes logic
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Treatment',
        hover_data=['Plate', 'Effective_Annotation'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    # Legend De-duplication
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Styling
    fig.update_traces(marker=dict(opacity=1.0)) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='x'))
    fig.update_traces(marker=dict(size=9, line=dict(width=1, color='black')), selector=dict(marker_symbol='square'))
    
    # --- FIXED AXIS (No Zero Lines) ---
    fig.update_layout(
        width=1300, height=850,
        legend_title_text='Pathway / Group',
        annotations=[
            dict(
                text="<b>Key:</b> Square = no_sgrna | Cross (x) = T1 | Dot (●) = T2/T0",
                showarrow=False, xref="paper", yref="paper",
                x=0.5, y=1.07, font=dict(size=14),
                bgcolor="white", bordercolor="black", borderwidth=1
            )
        ],
        xaxis=dict(
            title="UMAP 1", showline=True, linewidth=2, linecolor='black', 
            mirror=False, showgrid=False, zeroline=False
        ),
        yaxis=dict(
            title="UMAP 2", showline=True, linewidth=2, linecolor='black', 
            mirror=False, showgrid=False, zeroline=False
        )
    )
    
    # Save Outputs
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{file_base}.svg"))

print(f"Success. All outputs saved to {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_5_annotatedT2")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. PLATE/TIME FILTERING ---
# SPECIFY YOUR PLATES HERE. Leave as None or [] to plot everything.
SELECTED_PLATES = ["PLATE1_T2", "PLATE2_T2", "PLATE3_T2","PLATE4_T2","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()
    print(f"Filtering complete. Plotting {len(df)} cells from {len(SELECTED_PLATES)} plates.")

# --- 3. MERGE & FALLBACK ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

# FALLBACK: Annotation 4 -> Annotation 3 -> Unknown
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3'])
df['Effective_Annotation'] = df['Effective_Annotation'].fillna("Unknown/Other")

df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Define the Display Category
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']

# Set Baseline/Control logic
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# --- 4. DYNAMIC COLOR MAPPING ---
# Expanded palette to avoid purple confusion
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
standard_colors = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['no_sgrna'] = 'lightgrey'
color_map['Baseline (T0)'] = '#D3D3D3'
color_map['Unknown/Other'] = 'black'

# --- 5. EXECUTION FOR ALL CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Define shapes logic
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Treatment',
        hover_data=['Plate', 'Effective_Annotation'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    # Legend De-duplication
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Styling
    fig.update_traces(marker=dict(opacity=1.0)) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='x'))
    fig.update_traces(marker=dict(size=9, line=dict(width=1, color='black')), selector=dict(marker_symbol='square'))
    
    # --- FIXED AXIS (No Zero Lines) ---
    fig.update_layout(
        width=1300, height=850,
        legend_title_text='Pathway / Group',
        annotations=[
            dict(
                text="<b>Key:</b> Square = no_sgrna | Cross (x) = T1 | Dot (●) = T2/T0",
                showarrow=False, xref="paper", yref="paper",
                x=0.5, y=1.07, font=dict(size=14),
                bgcolor="white", bordercolor="black", borderwidth=1
            )
        ],
        xaxis=dict(
            title="UMAP 1", showline=True, linewidth=2, linecolor='black', 
            mirror=False, showgrid=False, zeroline=False
        ),
        yaxis=dict(
            title="UMAP 2", showline=True, linewidth=2, linecolor='black', 
            mirror=False, showgrid=False, zeroline=False
        )
    )
    
    # Save Outputs
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{file_base}.svg"))

print(f"Success. All outputs saved to {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_5_annotated3and4names")
COORD_DIR = os.path.join(OUTPUT_DIR, "Coordinates")

for folder in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. PLATE/TIME FILTERING ---
SELECTED_PLATES = [] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# --- 3. MERGE & FALLBACK ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3']).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# --- 4. DYNAMIC COLOR MAPPING ---
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
standard_colors = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['no_sgrna'] = 'lightgrey'
color_map['Baseline (T0)'] = '#D3D3D3'
color_map['Unknown/Other'] = 'black'

# --- 5. EXECUTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Save coordinates
    coord_filename = f"Coordinates_{config['name']}.csv"
    df[['Plate', 'Well_ID', 'Treatment', 'Display_Category', 'UMAP1', 'UMAP2']].to_csv(os.path.join(COORD_DIR, coord_filename), index=False)
    
    # --- 6. PLOTTING ---
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        text='Treatment', 
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'Effective_Annotation': True, 'UMAP1': False, 'UMAP2': False},
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Labels kept at size 16
    fig.update_traces(
        mode='markers+text', 
        textposition='top center', 
        textfont=dict(size=16, color='black'), 
        marker=dict(opacity=1.0)
    )
    
    # UPDATED: Marker sizes are now exactly twice the previous version
    fig.update_traces(marker=dict(size=10), selector=dict(marker_symbol='circle')) # Was 5
    fig.update_traces(marker=dict(size=14), selector=dict(marker_symbol='x'))      # Was 7
    fig.update_traces(marker=dict(size=18, line=dict(width=1, color='black')), selector=dict(marker_symbol='square')) # Was 9
    
    # Axis titles kept at size 18
    fig.update_layout(
        width=1600, height=1000,
        legend_title_text='Pathway / Group',
        annotations=[
            dict(
                text="<b>Key:</b> Square = no_sgrna | Cross (x) = T1 | Dot (●) = T2/T0",
                showarrow=False, xref="paper", yref="paper",
                x=0.5, y=1.07, font=dict(size=14),
                bgcolor="white", bordercolor="black", borderwidth=1
            )
        ],
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=18)), 
            showline=True, linewidth=2, linecolor='black', 
            mirror=False, showgrid=False, zeroline=False
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=18)), 
            showline=True, linewidth=2, linecolor='black', 
            mirror=False, showgrid=False, zeroline=False
        )
    )
    
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{file_base}.svg"))

print(f"Done. Markers (Dots: 10, Crosses: 14), Labels (16), and Axis Titles (18) are exported.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_5_annotated3and4namesandwell")
COORD_DIR = os.path.join(OUTPUT_DIR, "Coordinates")

for folder in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. PLATE/TIME FILTERING ---
SELECTED_PLATES = [] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# --- 3. MERGE & FALLBACK ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3']).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Define the Display Category
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# Create a combined label for the plot: "Mutant (Well)"
df['Plot_Label'] = df['Treatment'] + " (" + df['Well_ID'].astype(str) + ")"

# --- 4. DYNAMIC COLOR MAPPING ---
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
standard_colors = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['no_sgrna'] = 'lightgrey'
color_map['Baseline (T0)'] = '#D3D3D3'
color_map['Unknown/Other'] = 'black'

# --- 5. EXECUTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Save coordinates
    coord_filename = f"Coordinates_{config['name']}.csv"
    df[['Plate', 'Well_ID', 'Treatment', 'Display_Category', 'UMAP1', 'UMAP2']].to_csv(os.path.join(COORD_DIR, coord_filename), index=False)
    
    # --- 6. PLOTTING ---
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        text='Plot_Label', # Displays "Mutant (Well)" next to the point
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Plot_Label',
        hover_data={'Plate': True, 'Well_ID': True, 'Effective_Annotation': True, 'UMAP1': False, 'UMAP2': False},
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Markers+text ensures the combined label is visible
    fig.update_traces(mode='markers+text', textposition='top center', textfont=dict(size=8), marker=dict(opacity=1.0)) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='x'))
    fig.update_traces(marker=dict(size=9, line=dict(width=1, color='black')), selector=dict(marker_symbol='square'))
    
    fig.update_layout(
        width=1400, height=900,
        legend_title_text='Pathway / Group',
        annotations=[
            dict(
                text="<b>Key:</b> Square = no_sgrna | Cross (x) = T1 | Dot (●) = T2/T0",
                showarrow=False, xref="paper", yref="paper",
                x=0.5, y=1.07, font=dict(size=14),
                bgcolor="white", bordercolor="black", borderwidth=1
            )
        ],
        xaxis=dict(title="UMAP 1", showline=True, linewidth=2, linecolor='black', mirror=False, showgrid=False, zeroline=False),
        yaxis=dict(title="UMAP 2", showline=True, linewidth=2, linecolor='black', mirror=False, showgrid=False, zeroline=False)
    )
    
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{file_base}.svg"))

print(f"Done. Labels now show 'Mutant (Well ID)'.")